# TB Portals â€” 01 Â· Build manifest

Day 1. Produces `manifest.csv` with the fixed contract columns: `image_id, image_path, patient_id, country, alp_0_100, cavity`.

- **Real data:** edit `COLUMN_MAP` below to your actual TB Portals reads column names (passed inline â€” no source edit needed).
- **No data yet:** run the *synthetic* cell to exercise the whole stack now.

In [ ]:
# ── Pull latest codebase from GitHub ─────────────────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"

if os.path.isdir(REPO_DIR):
subprocess.run(["git","-C",REPO_DIR,"pull","--ff-only"],check=True)
else:
subprocess.run(["git","clone","--depth","1","--branch","cleaned-repo","https://github.com/mabdullahi7780/dl-project-codebase.git",REPO_DIR],check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo ready at", REPO_DIR)


In [ ]:
# --- Clone / update the repo (requires Internet enabled in Kaggle) ---
import os, subprocess, sys

REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"

if os.path.exists(REPO_DIR):
subprocess.run(["git","-C",REPO_DIR,"pull","--ff-only"],check=True)
else:
subprocess.run(["git","clone","--depth","1","--branch","cleaned-repo","https://github.com/mabdullahi7780/dl-project-codebase.git",REPO_DIR],check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo ready at", REPO_DIR)

In [ ]:
import sys, os, pandas as pd

REPO_DIR    = '/kaggle/working/dl-project-codebase'
DATASET_DIR = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs/kaggle_export'
WORK        = '/kaggle/working'
sys.path.insert(0, REPO_DIR)
os.makedirs(f'{WORK}/data/processed', exist_ok=True)

MANIFEST = f'{WORK}/data/processed/tbportals_manifest.csv'
print('python', sys.version.split()[0])
print('images at: /kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs/kaggle_export/images')


## Load manifest from uploaded PNG dataset

The `tb-portals-cxr-pngs` Kaggle dataset was created locally by running:
```
python scripts/export_kaggle_dataset.py \
    --manifest local_work/data/processed/tbportals_manifest.csv \
    --out-dir  local_work/kaggle_export
```
It contains `images/*.png` (512x512 grayscale) and `manifest.csv` with relative paths.
This cell rewrites `image_path` to absolute Kaggle paths.


In [ ]:
import sys, os
REPO_DIR = '/kaggle/working/dl-project-codebase'
# Ensure repo is cloned and on sys.path
if not os.path.isdir(REPO_DIR):
    import subprocess
subprocess.run(["git","clone","--depth","1","--branch","cleaned-repo","https://github.com/mabdullahi7780/dl-project-codebase.git",REPO_DIR],check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.data.tbportals import load_manifest, summarize_manifest

# Load pre-built manifest from the uploaded Kaggle dataset
_raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv',
                   dtype={'image_id': str, 'patient_id': str, 'country': str})

# Rewrite relative paths (images/<stem>.png) -> absolute Kaggle paths
_raw['image_path'] = _raw['image_path'].apply(
    lambda p: f"{KAGGLE_EXPORT}/{p}" if isinstance(p, str) else p
)

_raw.to_csv(MANIFEST, index=False)
df = load_manifest(MANIFEST)
print(f'Loaded {len(df)} images')
summarize_manifest(df)


## Option B — synthetic dataset (no data uploaded yet)
Skip this cell if the real manifest loaded above without errors.


In [ ]:
import sys, os
REPO_DIR = '/kaggle/working/dl-project-codebase'
# Ensure repo is cloned and on sys.path
if not os.path.isdir(REPO_DIR):
    import subprocess
subprocess.run(["git","clone","--depth","1","--branch","cleaned-repo","https://github.com/mabdullahi7780/dl-project-codebase.git",REPO_DIR],check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.data.tbportals import make_synthetic_dataset, load_manifest, summarize_manifest
MANIFEST = str(make_synthetic_dataset(f'{WORK}/data/processed/synth', n=600, seed=0))

## Verify

In [ ]:
df = load_manifest(MANIFEST)
summarize_manifest(df)
for c in ['Romania', 'Moldova', 'Kazakhstan']:
    assert c in set(df['country']), f'Held-out country missing: {c}'
print('\nManifest OK ->', MANIFEST)